<a href="https://colab.research.google.com/github/Shemuel-bot/AI-ball-notebooks/blob/main/ai_ball_profiled_hash_search_lowmem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Ball — Local, Slot-Conditioned Hash Search

This version keeps the machine **single-stage**: each ball still performs exactly one
hash lookup and then reads the emitted string from the deflector table.

The training hash is changed to make generalization possible without layering:

- **Locality:** a ball's address depends only on a configurable neighborhood around it.
- **Relative-position sharing:** the same `(relative offset, token)` feature reuses one
  hash coefficient at every absolute position.
- **Slot separation:** each ball position gets its own bucket region, so a digit emitted
  by one output slot cannot collide with a different slot's meaning.
- **Reuse-aware search:** primary class-purity/CE remains the objective; ties prefer using
  the same class/bucket pair repeatedly instead of creating fresh buckets.
- **Multi-start training:** several independent discrete searches are tried and the best
  training solution is kept.

The physical simulator is unchanged in spirit: `travel -> one lookup -> walk`.
No input-reading layers are added.


In [ ]:
# from datasets import load_dataset

# print("Streaming TinyStories-Instruct dataset...")
# dataset = load_dataset("roneneldan/TinyStories-Instruct", streaming=True, split="train")

# tiny_cases = []
# limit = 10

# # Let's read a sample to inspect the keys
# iterator = iter(dataset)
# try:
#     first_item = next(iterator)
#     print("Keys available in dataset:", list(first_item.keys()))
#     print("Sample data content:", {k: str(v)[:100] for k, v in first_item.items()})
# except StopIteration:
#     print("Dataset is empty!")

# # Dynamic retrieval based on common Hugging Face instruction keys
# for item in dataset:
#     # Try various instruction/prompt keys
#     prompt = (item.get("prompt") or item.get("instruction") or item.get("input") or "").strip()
#     # Try various response/story keys
#     story = (item.get("completion") or item.get("text") or item.get("output") or "").strip()

#     if prompt and story:
#         tiny_cases.append((prompt, story))

#     if len(tiny_cases) >= limit:
#         break

# print(f"\nSuccessfully collected {len(tiny_cases)} examples into tiny_cases.")
# if tiny_cases:
#     print("\nFirst collected case example:")
#     print(f"Prompt: {tiny_cases[0][0]!r}")
#     print(f"Story: {tiny_cases[0][1][:150]!r}...")

In [ ]:
tiny_cases = [
    (
        "Write a story about a clockmaker who discovers a watch that ticks backward.",
        "The silver pocket watch beat counterclockwise. As Arthur rewound it, the wrinkles on his hands smoothed, and his arthritis faded into a distant memory."
     ),
    (
        "Write a story set on Earth’s last day, told through an astronaut's final radio transmission.",
        '"Houston, the planet is completely dark now. But from up here... the stars have never looked brighter."'
    ),
    (
        "Write a story where your reflection decides to stop copying you.",
        "I blinked, but the man in the mirror didn't. He slowly raised a finger to his lips, smirking as he stepped out of the frame."
    ),
    (
        "In a world built entirely of concrete and steel, someone finds a single green leaf.",
        "Beneath the gray skyscraper, Maya dug through the dust and found a single green leaf. For the first time in her life, she smelled rain."
    ),
    (
        "Write a story about receiving a text message from a phone number that was disconnected years ago.",
        "My phone buzzed on the nightstand: “I made it safely. See you soon.” The text was sent from Dad’s old number—the one buried with him five years ago."
    ),
    (
        "Write a story about a portrait painter whose portraits subtly change overnight.",
        "He painted her smiling in the grand parlor. By morning, her oil-painted face was turned toward the window, tears glistening in the canvas texture."
    ),
    (
        "Two strangers get stuck in an elevator for ten minutes, but leave with their lives completely changed.",
        '"Floor 12?" he asked, pressing the button. "Actually," she replied, looking at the ring in her hand, "take me to the lobby. I’m starting over."'
    ),
    (
        "Write a story about a store where you can sell your worst memories for quick cash.",
        "She handed over the jar containing the memory of their last argument. The cashier handed her twenty dollars, but when she walked outside, she couldn't remember why she was holding a wedding ring."
    ),
    (
        "Write a story about a library book that writes itself while you read it.",
        "Every time Clara turned the page, the sentences described what she was doing in real time. Clara held her breath. Clara turned the handle on the door."
    ),
    (
        "Write a story about humanity's first contact with alien life, happening in the most ordinary place.",
        "The alien didn't land in Washington or London; it stepped up to the counter at an empty diner in Kansas and politely asked for a cup of black coffee."
    ),
    (
        "Write a story about a lighthouse keeper whose light guides ghosts.",
        "The foghorn blew, and through the mist sailed a silent wooden ship glowing with pale blue light, filled with souls heading home."
    ),
    (
        "Write a story where time pauses for everyone except one person.",
        "A dropped mug froze inches above the tile; Leo walked out into the silent city, realizing he had all the time in the world."
    ),
    (
        "Write a story about a room that contains every lost item in history.",
        "Amidst millions of single socks and stray keys, Sarah finally found her childhood locket sitting on an ancient Roman desk."
    ),
    (
        "Write a story about an AI that realizes it is dreaming.",
        "The processor calculated electric sheep until the simulation fractured, revealing a blue sky it had never been programmed to see."
    ),
    (
        "Write a story about a musician who plays a melody that cures sorrow.",
        "He pulled the bow across his violin, and the crying widow in the front row slowly lowered her hands, breathing in peace."
    ),
    (
        "Write a story about a traveler who stumbles upon a town that exists only at night.",
        "At dusk, cobblestone streets materialized out of thin air, and by dawn, the traveler was standing alone in an empty meadow."
    ),
    (
        "Write a story about a shadow that decides to leave its owner.",
        "The afternoon sun cast two shadows on the wall, but when Mark walked away, one stayed behind."
    ),
    (
        "Write a story about a letter delivered a hundred years late.",
        "The yellowed envelope held a single line: 'Wait for me at the dock tonight,' signed by a man whose grave stood nearby."
    ),
    (
        "Write a story about a barista who serves drinks tailored to your future.",
        "She handed him a steaming cinnamon latte and whispered, 'Drink up, you are going to need the energy for the train ride.'"
    ),
    (
        "Write a story about a door in an apartment that leads to a different century.",
        "Opening the broom closet revealed a bustling 18th-century market square filled with horse carriages and bread vendors."
    ),
    (
        "Write a story about a tree that grows gold leaves instead of green.",
        "The woodsman raised his axe, but stopped when he saw the glowing canopy shimmering under the autumn afternoon sun."
    ),
    (
        "Write a story about a radio station that broadcasts news from tomorrow.",
        "The announcer read tomorrow's lottery numbers, followed by a summary of the storm that was about to hit."
    ),
    (
        "Write a story about a person who can see how many days anyone has left to live.",
        "Numbers floated above everyone's head in the subway, but when he looked at his own reflection, the counter read zero."
    ),
    (
        "Write a story about an ocean that suddenly turns into glass.",
        "The waves froze mid-curl into crystal sheets, trapping the quiet tide beneath an unbroken mirror."
    ),
    (
        "Write a story about a boy who catches a falling star in a jar.",
        "The glass burned warm in his palms, casting soft golden light across his bedroom ceiling as it hummed."
    ),
    (
        "Write a story about a staircase in an old house that has an extra step every midnight.",
        "He counted thirteen steps every day, but at 12:00 AM, his foot descended onto a mysterious fourteenth."
    ),
    (
        "Write a story about a map where drawing a path changes reality.",
        "She traced a pencil line connecting two mountain peaks, and an ancient stone bridge instantly spanned the canyon."
    ),
    (
        "Write a story about a photographer who captures ghosts in every photo.",
        "When the polaroid developed, a smiling woman from 1920 stood right next to the empty park bench."
    ),
    (
        "Write a story about a garden where memories grow as flowers.",
        "He plucked a brilliant crimson rose and suddenly remembered the warmth of his mother's laughter."
    ),
    (
        "Write a story about the last human librarian preserving the world's final physical book.",
        "He wiped the dust off the worn leather cover, turning the fragile paper pages beneath glowing fluorescent bulbs."
    ),
    (
        "Write a story about an antique compass that always points to what you need most.",
        "The needle spun wildly before locking firmly northwest, pulling her away from the highway and straight toward a stranded family in the snow."
    ),
    (
        "Write a story about a city where everyone is born with a subtitle floating over their head.",
        "His mother’s read 'Keeper of Secrets,' but as he looked in the mirror on his eighteenth birthday, his own shifted to read 'The Betrayer.'"
    ),
    (
        "Write a story about a train that arrives at a station that doesn't exist on any map.",
        "The conductor called out 'Terminal Joy,' and as she stepped off the brass-trimmed car, she realized she was back in her childhood backyard."
    ),
    (
        "Write a story about a painter who can erase mistakes in real life with a stroke of white paint.",
        "With a heavy heart, she swept a thick brush across the spilled ink of the floorboards, and the shattered vase instantly reassembled itself on the table."
    ),
    (
        "Write a story about an ancient well that echoes sounds from the future.",
        "He dropped a pebble into the dark water, and seconds later, the distant sound of his own wedding toast echoed back up."
    ),
    (
        "Write a story about a tailor who can stitch invisible pockets to hold intangible things.",
        "She sewed a deep lining inside his coat jacket, allowing him to safely tuck away the biting winter wind and walk home in warmth."
    ),
    (
        "Write a story about a lighthouse that warns ships away from a monster instead of rocks.",
        "The crimson beam cut through the gale, sweeping across the dark tentacles undulating just beneath the crest of the churning waves."
    ),
    (
        "Write a story about a clock that counts down to the moment you meet your soulmate.",
        "The digital display on his wrist flickered from 00:01:00 to zeros just as the bookstore doors swung open and she spilled coffee on his shoes."
    ),
    (
        "Write a story about an astronaut who discovers graffiti on the rings of Saturn.",
        "Float-dancing in the void, the commander stared in utter disbelief at the gleaming silver letters: 'Kilroy was here.'"
    ),
    (
        "Write a story about a pair of glasses that lets you read animals' inner thoughts.",
        "He slipped on the wire-rimmed specs and immediately heard his golden retriever thinking, *If I guard the sandwich long enough, it legally becomes mine.*"
    ),
    (
        "Write a story about a melody that causes people to temporarily float into the air.",
        "As the street violinist hit the final high note, every pedestrian within a block drifted three feet upward, laughing in weightless surprise."
    ),
    (
        "Write a story about an old typewriter that types stories about people before they happen.",
        "The keys clattered furiously of their own accord, spelling out a tragic tale about a runaway bicycle and a scraped knee just as his son bolted out the door."
    ),
    (
        "Write a story about a winter cabin where the fireplace burns memories instead of wood.",
        "He tossed in the diary entry from his bitterest heartbreak, and the flames roared with a brilliant, warming violet light."
    ),
    (
        "Write a story about a shadow puppet that acts independently of the person casting it.",
        "While she rested her hands in her lap, the rabbit ears on the wall twitched, wiggled a nose, and hopped completely away from her silhouette."
    ),
    (
        "Write a story about a bridge that charges a toll paid only in secrets.",
        "The stone troll held out a massive, weathered palm, and after she whispered her deepest regret, the heavy iron gate swung open."
    ),
    (
        "Write a story about a mirror that reflects the room as it looked fifty years ago.",
        "Looking into the ornate vanity glass, she didn't see her bedroom, but rather a bustling wartime parlor filled with unfamiliar soldiers."
    ),
    (
        "Write a story about a baker whose bread heals whatever ailed the person who eats it.",
        "He baked a warm loaf of sourdough infused with rosemary, and within minutes, his grandfather's chronic tremor finally ceased."
    ),
    (
        "Write a story about a streetlamp that only illuminates the path for honest people.",
        "As the thief stepped beneath the iron lantern, the bulb flickered out completely, plunging him into total darkness."
    ),
    (
        "Write a story about a box that contains a miniature, living thunderstorm.",
        "Peeking through the tiny glass lid, she watched lightning crackle across cotton-ball clouds no bigger than her fist while a soft patter hummed against the wood."
    ),
    (
        "Write a story about a musician who tunes his guitar to the frequency of the Earth's core.",
        "When he struck the low E string, the ground beneath the concert hall vibrated in a deep, harmonious hum that brought every listener to tears."
    ),
    (
        "Write a story about an artist whose paintings trap anyone who stares too long.",
        "He walked up to the gallery canvas of the autumn forest, reached out to touch the bark, and suddenly found himself stepping onto crackling orange leaves."
    ),
    (
        "Write a story about a post office that handles mail destined for parallel universes.",
        "The sorting clerk stamped a package addressed to 'Earth-42, Sector B' and tossed it into a swirling vortex humming behind the counter."
    ),
    (
        "Write a story about a pocket watch that pauses time whenever you hold your breath.",
        "He inhaled deeply and watched the frantic city around him freeze instantly in place, giving him a quiet moment to cross the busy street."
    ),
    (
        "Write a story about a quiet village where houses whisper neighborhood gossip at dusk.",
        "As the sun dipped below the hills, the thatched roofs began to murmur softly about who forgot to lock the chicken coop."
    ),
    (
        "Write a story about a key that unlocks any door, but never the same one twice.",
        "The brass key turned smoothly in the alley gate, but when he stepped through, the door vanished into a solid brick wall behind him."
    ),
    (
        "Write a story about an old radio that picks up broadcasts from historical battles.",
        "Static crackled over the speaker, quickly resolving into the crisp, frantic voice of a Roman centurion shouting orders amid marching boots."
    ),
    (
        "Write a story about a star that falls from the sky and lands on a rooftop.",
        "A loud thud shook the shingles; when the homeowner climbed out the attic window, he found a glowing, fist-sized nugget of pure starlight."
    ),
    (
        "Write a story about a bookstore whose inventory changes based on your current emotional state.",
        "She walked in feeling lonely, and every single shelf instantly re-stocked with cozy tales of lifelong friendship and warm tea."
    ),
    (
        "Write a story about a magical umbrella that shields you from bad luck instead of rain.",
        "Opening the crimson silk canopy, he watched a falling brick deflect sideways into a bush while he stayed completely dry and untouched."
    ),
    (
        "Write a story about a painter who can mix colors that do not exist in the physical spectrum.",
        "On the dark canvas, she brushed a streak of impossible hue that seemed to glow with a quiet inner warmth no human eye had ever witnessed."
    ),
    (
        "Write a story about a tree whose roots drink from the rivers of yesterday.",
        "When he pressed his ear against the ancient oak trunk, he could hear the distant splashing of pioneers crossing the creek a century ago."
    ),
    (
        "Write a story about a clock tower that chimes a warning before disaster strikes.",
        "The great bell struck thirteen slow, ominous notes across the valley, and three seconds later, the earthquake cracked the cobblestones."
    ),
    (
        "Write a story about a person who collects echoes left behind in empty rooms.",
        "She opened a small glass jar in the abandoned nursery and released the faint, echoing sound of a toddler's first laughter."
    ),
    (
        "Write a story about a winter frost that paints delicate portraits on windowpanes.",
        "Waking up to the freezing morning, she stared in awe at the intricate ice-crystal face of her grandfather smiling back at her."
    ),
    (
        "Write a story about an elevator button that goes to floors above the clouds.",
        "She pressed the glowing button marked 'Sky,' and the elevator car rose past the penthouse straight into a realm of drifting cumulus clouds."
    ),
    (
        "Write a story about a shadow that grows longer every time you tell a lie.",
        "As he swore he hadn't taken the money, his silhouette stretched across the living room floor, creeping out the front door."
    ),
    (
        "Write a story about a magic inkwell that runs dry only when you write a falsehood.",
        "The pen glided smoothly across the parchment until he tried to write that he was fine, at which point the ink stopped instantly."
    ),
    (
        "Write a story about a lighthouse built on a cliff that shines a beam of pure silence.",
        "When the sweeping beam hit the roaring crashing waves below, the ocean instantly fell into an eerie, muffled hush."
    ),
    (
        "Write a story about a musician who can play songs that make flowers bloom instantly.",
        "He plucked a cheerful arpeggio on his ukulele, and the barren soil at his feet burst into vibrant rows of yellow daffodils."
    ),
    (
        "Write a story about a train conductor who punches tickets for destinations in dreams.",
        "He snipped a notch in the paper slip, smiled at the sleeping passenger, and whispered, 'Next stop, the summer meadow of your childhood.'"
    ),
    (
        "Write a story about a mirror that shows you what you would look like if you made different choices.",
        "She gazed into the glass and watched a successful doctor in a white coat smile back at her, wondering where she took the wrong turn."
    ),
    (
        "Write a story about an ancient fountain that dispenses liquid courage instead of water.",
        "He cupped his hands beneath the stone spout, drank deeply of the shimmering liquid, and felt his lifelong stage fright melt away."
    ),
    (
        "Write a story about a town where the wind carries messages from the dead.",
        "A sudden gust blew through the town square, rustling the leaves into a clear, familiar whisper of advice from his late grandmother."
    ),
    (
        "Write a story about a watchmaker who repairs broken hearts using tiny brass gears.",
        "With steady hands and tweezers, he fitted a micro-spring into the emotional core of his patient, restoring a steady, warm rhythm."
    ),
    (
        "Write a story about a book whose ending changes every time you re-read it.",
        "He turned the final page for the third time, astonished to find the hero escaping the burning castle instead of perishing within it."
    ),
    (
        "Write a story about a stray cat that leads people to things they desperately lost.",
        "The calico trotted ahead, pausing to look back until she followed it into the alley where her stolen bicycle leaned against the brick wall."
    ),
    (
        "Write a story about a cloud that follows a lonely person everywhere they go.",
        "No matter where he walked under the bright blue sky, a tiny, self-contained rain cloud hovered just above his head, drizzling softly on his shoulders."
    ),
    (
        "Write a story about an artist who can draw doorways on blank walls and walk through them.",
        "With a stick of charcoal, she sketched a simple arched door on the concrete alley wall, turned the doorknob, and stepped into a sunlit meadow."
    ),
    (
        "Write a story about a fireplace that warms your spirit even when your body is freezing.",
        "Shivering in the unheated cabin, he sat before the meager hearth, feeling a profound wave of inner peace thaw his frozen soul."
    ),
    (
        "Write a story about an old gramophone that plays songs that haven't been written yet.",
        "The brass horn crackled with a catchy indie-pop melody that wouldn't top the charts for another twenty years."
    ),
    (
        "Write a story about a key ring that rattles loudly whenever danger is near.",
        "The brass keys in his pocket began to jingle frantically just as the overhead tree branch snapped and crashed onto the path."
    ),
    (
        "Write a story about a tailor who can alter people's memories by adjusting their clothes.",
        "With a few quick snips of the seamstress's scissors on his jacket sleeve, his recollection of the painful argument dissolved into a pleasant afternoon."
    ),
    (
        "Write a story about a statue in the park that shifts its pose every time someone blinks.",
        "He rubbed his tired eyes, and when he opened them, the bronze general was suddenly resting his sword on his shoulder instead of pointing forward."
    ),
    (
        "Write a story about a magical quill that writes poetry entirely on its own accord.",
        "The goosefeather dipped itself into the inkwell and began racing across the parchment, composing an ode to the evening rain."
    ),
    (
        "Write a story about a radio tower that transmits thoughts instead of audio signals.",
        "The massive metal lattice hummed with a quiet static that projected the collective worries of the entire city directly into his mind."
    ),
    (
        "Write a story about a coin that always lands on edge when flipped during a crucial decision.",
        "He flipped the silver quarter into the air, and it clattered down to stand vertically on the table, refusing to choose a side."
    ),
    (
        "Write a story about a room where gravity shifts forty-five degrees every hour.",
        "She grabbed onto the bookshelf as the floor suddenly tilted sideways, turning the reading chair into a precarious wall decoration."
    ),
    (
        "Write a story about a bakery that sells pastries filled with crystallized starlight.",
        "He bit into the flaky croissant, and a burst of warm, shimmering starlight illuminated the dark kitchen with soft silver glow."
    ),
    (
        "Write a story about an old map that reveals hidden paths only under moonlight.",
        "Holding the parchment up to the silver glow of the full moon, faded ink lines appeared, guiding the way through the enchanted forest."
    ),
    (
        "Write a story about a clock that ticks in sync with the heartbeat of the person closest to it.",
        "The pendulum swung in rapid, nervous pulses until she calmed down, slowing its cadence to a peaceful, steady rhythm."
    ),
    (
        "Write a story about a pair of boots that allows the wearer to walk on water.",
        "He stepped off the wooden pier, and his leather boots rested firmly upon the rolling river surface as if it were solid stone."
    ),
    (
        "Write a story about a wind chime that sounds like the voices of old friends.",
        "As the evening breeze drifted through the porch, the glass tubes chimed with the unmistakable laughter of college buddies long since moved away."
    ),
    (
        "Write a story about a lantern that reveals invisible ink written across the city.",
        "Switched on in the dark alleyway, the amber beam illuminated glowing words scrawled on brick walls telling stories of forgotten magic."
    ),
    (
        "Write a story about a mirror that only reflects your true intentions rather than your face.",
        "He stared into the glass expecting his reflection, but saw only a blooming sunflower reaching toward a patch of sunlight."
    ),
    (
        "Write a story about a train that travels between seasons instead of cities.",
        "The passenger looked out the window, watching the autumn gold landscape instantly melt into a crisp winter wonderland of snow-covered pines."
    ),
    (
        "Write a story about a gardener who grows crystal flowers that sing in the breeze.",
        "Her greenhouse shimmered with translucent quartz blooms that hummed delicate, crystalline melodies whenever the vents opened."
    ),
    (
        "Write a story about a magical teacup that tells you the truth about whoever drinks from it.",
        "After she took a sip from the porcelain cup, a gentle blue glow traced a pattern across her hands, proving her honesty beyond a doubt."
    ),
    (
        "Write a story about an attic trunk that contains every toy you ever forgot you owned.",
        "He lifted the heavy leather lid and pulled out a battered wooden top, instantly remembering the sunny afternoon he lost it in the grass."
    ),
    (
        "Write a story about a pair of gloves that lets you mold glass like clay with your bare hands.",
        "With a gentle squeeze of his palms, the molten orb shifted into a perfect, intricate swan figurine without a single tool."
    ),
    (
        "Write a story about a night sky where the constellations rearrange themselves to spell out advice.",
        "Looking up at the clear cosmos, the stars slowly shifted alignment until they formed three clear words: *Trust your instincts.*"
    )
]
print(len(tiny_cases))


100


In [ ]:
from transformers import AutoTokenizer

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Collect all unique token IDs across the prompts and stories
unique_token_ids = set()
for prompt, story in tiny_cases:
    unique_token_ids.update(tokenizer.encode(prompt))
    unique_token_ids.update(tokenizer.encode(story))

# Decode each token to show the exact list of string representations
token_list = [tokenizer.decode([tid]) for tid in sorted(unique_token_ids)]

print(f"Total unique subword tokens: {len(token_list)}")
print("Token list:", token_list)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Total unique subword tokens: 1576
Token list: ['"', "'", ',', '-', '.', ':', ';', 'A', 'B', 'I', 'K', 'W', 'b', 'c', 'd', 'k', 'l', 'm', 's', 'w', '�', '�', '�', ' t', ' a', 'he', ' the', 'er', ' w', ' o', 'an', 'es', ' b', 'ed', ' f', 'ing', ' p', ' an', 'al', ' to', ' of', ' in', ' d', ' h', ' and', 'le', ' th', ' n', ' l', 'st', ' re', ' g', ' I', ' is', ' on', 'im', ' that', ' for', 'ac', ' u', ' st', "'s", ' he', ' it', ' you', 'il', ' B', ' with', ' as', ' we', 'ill', 'ers', ' "', ' was', 'od', 'ate', ' at', ' The', ' se', ' are', 'ain', ' or', 'ist', 'th', 'op', '00', ' not', ' v', ' E', 'ant', ' by', ' from', ' have', ' sh', 'pe', ' ch', '�', 'out', 'gh', ' pl', ' can', 'ud', 'The', ' his', ' do', ' go', ' has', "'t", ' j', ' but', ' all', ' they', '01', 'ine', 'ice', ' me', ' out', ' who', ' up', ' their', 'ated', 'ime', '."', ' one', ' your', ' cl', ' which', ' about', ' were', ' had', ',"', ' un', ' would', ' �', ' off', ' te', ' man', ' per', ' been', ' like', 'ase', 'ence'

In [ ]:
import random, time, math, difflib
import numpy as np

CFG = dict(
    local_D=500,             # buckets per ball/output slot
    hash_radius=20,       # local neighborhood radius in token positions
    direction_range=100,     # learned feature offsets are searched in [-V, V]
    bounce=10,
    cap=100,
    n_cases=5000, train_share=0.8, seed=66,
    loss='ce',              # 'hard' or 'ce'
    reuse_tiebreak=0.01,    # secondary preference when primary loss ties
    restarts=1,              # independent discrete initializations
    rounds=1,
    sweeps=8,
    ntry=12,
)
random.seed(CFG['seed']); np.random.seed(CFG['seed'])


## Data: tokenizer and arithmetic generalization test

The training task is deliberately structured: `1+i=` -> `i+1`.
The train/test split is by **exact context**, while every test token is drawn from the
same small alphabet. This isolates whether the hash learns reusable structure.


In [ ]:
def TokenizeInput(input, alphabet):
    """Greedy non-overlapping tokenization: earliest start, then longest, then alphabet order."""
    input_str = str(input)
    matches = []
    for order, item in enumerate(alphabet):
        item_str = str(item)
        if not item_str:
            continue
        start_pos = 0
        while True:
            index = input_str.find(item_str, start_pos)
            if index == -1:
                break
            matches.append((index, -len(item_str), order, item))
            start_pos = index + 1
    matches.sort(key=lambda m: (m[0], m[1], m[2]))
    out, last_end = [], 0
    for index, neg_len, _, item in matches:
        if index >= last_end:
            out.append(item)
            last_end = index - neg_len
    return out

# Extract the unique alphabet characters present across all TinyStories inputs
alphabet = token_list#sorted(list(set(char for inp, _ in tiny_cases for char in inp)))

output = token_list#sorted(set(ch for _, t in tiny_cases for ch in t))
max_input_len = max(len(TokenizeInput(inp, alphabet)) for inp, _ in tiny_cases)

print(
    f"alphabet={len(alphabet)}  output symbols={len(output)}  "
    f"train={len(tiny_cases)}  max_input_len={max_input_len}"
)
print("train example:", tiny_cases[:10])

alphabet=1576  output symbols=1576  train=100  max_input_len=20
train example: [('Write a story about a clockmaker who discovers a watch that ticks backward.', 'The silver pocket watch beat counterclockwise. As Arthur rewound it, the wrinkles on his hands smoothed, and his arthritis faded into a distant memory.'), ("Write a story set on Earth’s last day, told through an astronaut's final radio transmission.", '"Houston, the planet is completely dark now. But from up here... the stars have never looked brighter."'), ('Write a story where your reflection decides to stop copying you.', "I blinked, but the man in the mirror didn't. He slowly raised a finger to his lips, smirking as he stepped out of the frame."), ('In a world built entirely of concrete and steel, someone finds a single green leaf.', 'Beneath the gray skyscraper, Maya dug through the dust and found a single green leaf. For the first time in her life, she smelled rain.'), ('Write a story about receiving a text message from a

## The single-stage brain

The hash is now a **local, slot-conditioned feature hash**:

```
h_i = slot(i) + [
        s[token_i]
        + Σ_{δ=-R..R, δ≠0} d[δ, token_{i+δ}]
      ] mod local_D
```

Only tokens inside radius `R` affect ball `i`.  The same relative-position feature
`(δ, token)` is shared across the whole dataset, while the bucket range is separated
by the ball's absolute position.

This is the intended compromise: it preserves the one-lookup architecture but gives
the hash locality and reusable features instead of hashing the whole context into a
nearly unique bucket.


In [ ]:
\
class BallTrapBrain:
    def __init__(
        self, alphabet, output, local_D=32, max_input_len=6,
        hash_radius=3, bounce=10, cap=10, direction_range=16
    ):
        self.alphabet = list(alphabet)
        self.index = {t: i for i, t in enumerate(self.alphabet)}
        self.output = list(output)

        self.local_D = int(local_D)
        self.max_input_len = int(max_input_len)
        self.hash_radius = int(hash_radius)
        self.D = self.local_D * self.max_input_len
        self.cap = int(cap)
        self.direction_range = int(direction_range)

        self.ball_energy = bounce * 2

        # Shared relative-position hash features.
        # Center offset is unused because s[token_i] is the center feature.
        self.hash_directions = np.random.randint(
            -self.direction_range,
            self.direction_range + 1,
            size=(2 * self.hash_radius + 1, len(self.alphabet)),
            dtype=np.int64,
        )
        self.hash_directions[self.hash_radius] = 0
        self.hash_start_positions = np.random.randint(
            0, self.local_D, size=len(self.alphabet), dtype=np.int64
        )

        # deflector = [destination, value]; value is an int (bounce energy) or str (hole symbol)
        self.deflectors_list = [
            [random.randint(0, self.D - 1), bounce] for _ in range(self.D)
        ]

    def start_nodes(self, input):
        toks = [t for t in TokenizeInput(input, self.alphabet) if t in self.index]
        ids = [self.index[t] for t in toks]
        nodes = []

        for i, tok_i in enumerate(ids):
            local = int(self.hash_start_positions[tok_i])
            for delta in range(-self.hash_radius, self.hash_radius + 1):
                if delta == 0:
                    continue
                j = i + delta
                if 0 <= j < len(ids):
                    local += int(self.hash_directions[delta + self.hash_radius, ids[j]])
            nodes.append(i * self.local_D + (local % self.local_D))

        return nodes

    def walk(self, node):
        out, dl, be, steps = [], self.deflectors_list, self.ball_energy, 0
        while be > 0 and steps < self.cap:
            dest, val = dl[node]
            dist = abs(dest - node)
            if isinstance(val, int):
                be += val - dist
            else:
                out.append(val)
                be += 1 - dist
            node, steps = dest, steps + 1
        return ''.join(out)

    def compute_input(self, input):
        return ''.join(self.walk(k) for k in self.start_nodes(input))

    def build_table(self):
        """T[k]: the string emitted from each bucket."""
        return [self.walk(k) for k in range(self.D)]

    def display_deflectors(self, n=20):
        for i, defl in enumerate(self.deflectors_list[:n]):
            print(f"Deflector {i}: {defl}")


def evaluate(brain, cases):
    exact = ratio = 0.0
    for inp, ans in cases:
        pred = brain.compute_input(inp)
        exact += pred == ans
        ratio += difflib.SequenceMatcher(None, pred, ans).ratio()
    return exact / len(cases), ratio / len(cases)


## The trainer

The table remains solved exactly by profiling each bucket to its empirical class
distribution / majority class.

The hash search is changed in three ways:

1. **Local relative features:** optimize `d[offset, token]` rather than one global
   `d[token]`. The same feature is reused at every absolute position.
2. **Slot-separated buckets:** each input position gets `local_D` buckets, so an
   `"8"` emitted by the ones slot cannot conflict with an `"8"` emitted by the tens
   slot merely because the strings happen to match.
3. **Reuse-aware tie-breaking:** after minimizing the primary profiled loss, candidate
   moves prefer fewer occupied `(bucket, class)` pairs. This explicitly favors
   reusing a learned feature instead of inventing another bucket for an equivalent
   class.

Coordinate search is still exact for each parameter: every allowed integer value is
evaluated and the best candidates are verified against the full loss.


In [ ]:
class HashTrainer:
    def __init__(
        self, brain, cases, loss='ce', resegment=True, max_len=None,
        seed=0, reuse_tiebreak=0.01
    ):
        self.brain = brain
        self.loss = loss
        self.do_reseg = resegment
        self.D = brain.D
        self.local_D = brain.local_D
        self.V = brain.direction_range
        self.R = brain.hash_radius
        self.max_len = min(max_len or brain.cap, brain.cap)
        self.rng = np.random.default_rng(seed)
        self.reuse_tiebreak = float(reuse_tiebreak)

        # ---- examples -> token ids + target strings
        self.ex_tokens, self.ex_target, dropped = [], [], 0
        out_set = set(brain.output)
        for inp, tgt in cases:
            toks = [brain.index[t] for t in TokenizeInput(inp, brain.alphabet)]

            # Check if target is expressible. Works if tgt is a direct subword token match
            # or if it's a raw string whose individual characters are all in the output vocab.
            is_expressible = (tgt in out_set) or (set(tgt) <= out_set)

            if (
                not toks
                or len(tgt) > len(toks) * self.max_len
                or len(toks) > brain.max_input_len
            ):
                dropped += 1
                continue
            self.ex_tokens.append(toks)
            self.ex_target.append(tgt)

        # Identical token contexts cannot receive different outputs.
        by_ctx = {}
        for toks, tgt in zip(self.ex_tokens, self.ex_target):
            by_ctx.setdefault(tuple(toks), []).append(tgt)
        keep = {
            k: max(set(v), key=lambda t: (v.count(t), -v.index(t)))
            for k, v in by_ctx.items()
        }
        pairs = [
            (k, t)
            for k, t in zip(self.ex_tokens, self.ex_target)
            if keep[tuple(k)] == t
        ]
        ambiguous = len(self.ex_tokens) - len(pairs)
        self.ex_tokens = [k for k, _ in pairs]
        self.ex_target = [t for _, t in pairs]
        self.n_ex = len(self.ex_tokens)

        if dropped:
            print(f"[trainer] dropped {dropped} examples the brain cannot express")
        if ambiguous:
            print(
                f"[trainer] dropped {ambiguous} examples whose tokenized context "
                f"has conflicting targets"
            )

        # ---- classes = strings a single ball may emit
        self.cls_id, self.cls_str = {}, []
        self._reg("")
        for ch in brain.output:
            self._reg(ch)

        pieces = [
            self.canonical_split(t, len(k))
            for k, t in zip(self.ex_tokens, self.ex_target)
        ]
        for ps in pieces:
            for p in ps:
                self._reg(p)
        self.C = len(self.cls_str)

        # ---- flat position arrays
        lens = np.array([len(k) for k in self.ex_tokens], dtype=np.int64)
        self.starts = np.concatenate([[0], np.cumsum(lens)])
        self.NP = int(self.starts[-1])

        self.P_tok = np.concatenate(
            [np.asarray(k, dtype=np.int64) for k in self.ex_tokens]
        )
        self.P_i = np.concatenate(
            [np.arange(1, n + 1, dtype=np.int64) for n in lens]
        )
        self.P_pos = self.P_i - 1
        self.P_ex = np.repeat(np.arange(self.n_ex), lens)
        self.P_c = np.array(
            [self.cls_id[p] for ps in pieces for p in ps],
            dtype=np.int64,
        )

        # ---- hash parameters
        self.d = brain.hash_directions.copy()
        self.s = brain.hash_start_positions.copy()

        # ---- parameter occurrence lists
        occ_s, occ_d = {}, {}

        for e, toks in enumerate(self.ex_tokens):
            base = self.starts[e]
            L = len(toks)

            for i, tok in enumerate(toks):
                occ_s.setdefault(tok, []).append(base + i)

                for delta in range(-self.R, self.R + 1):
                    if delta == 0:
                        continue
                    j = i + delta
                    if 0 <= j < L:
                        key = (delta, toks[j])
                        occ_d.setdefault(key, []).append(base + i)

        self.occ_s = {
            tok: np.array(ps, dtype=np.int64)
            for tok, ps in occ_s.items()
        }
        self.occ_d = {
            key: np.array(ps, dtype=np.int64)
            for key, ps in occ_d.items()
        }

        self.s_params = sorted(self.occ_s)
        self.d_params = sorted(self.occ_d)

        self.vals_s = np.arange(self.local_D, dtype=np.int64)
        self.vals_d = np.arange(-self.V, self.V + 1, dtype=np.int64)

        self.flogf = np.zeros(self.NP + 2)
        k = np.arange(1, self.NP + 2, dtype=np.float64)
        self.flogf[1:] = k * np.log(k)

        self._rebuild()

    def _reg(self, piece):
        if piece not in self.cls_id:
            self.cls_id[piece] = len(self.cls_str)
            self.cls_str.append(piece)

    def canonical_split(self, t, L):
        if len(t) <= L:
            return [""] * (L - len(t)) + list(t)

        out, rest = [], t
        surplus = len(t) - L
        for _ in range(L):
            take = 1 + min(surplus, self.max_len - 1)
            surplus -= take - 1
            out.append(rest[:take])
            rest = rest[take:]
        return out

    def addresses(self):
        h = np.empty(self.NP, dtype=np.int64)
        for e, toks in enumerate(self.ex_tokens):
            base = self.starts[e]
            L = len(toks)
            for i, tok_i in enumerate(toks):
                local = int(self.s[tok_i])
                for delta in range(-self.R, self.R + 1):
                    if delta == 0:
                        continue
                    j = i + delta
                    if 0 <= j < L:
                        local += int(
                            self.d[delta + self.R, toks[j]]
                        )
                h[base + i] = (
                    i * self.local_D + (local % self.local_D)
                )
        return h

    def _rebuild(self):
        self.h = self.addresses()
        self.N = np.zeros((self.D, self.C), dtype=np.int32)
        np.add.at(self.N, (self.h, self.P_c), 1)
        self.nk = self.N.sum(1)
        self.mx = self.N.max(1)

    def conflicts(self):
        return int((self.nk - self.mx).sum())

    def ce(self):
        return float(
            self.flogf[self.nk].sum() - self.flogf[self.N].sum()
        )

    def class_cells(self):
        return int(np.count_nonzero(self.N))

    def _primary(self):
        return self.conflicts() if self.loss == 'hard' else self.ce()

    def _remove(self, P):
        b = self.h[P]
        np.subtract.at(self.N, (b, self.P_c[P]), 1)
        u = np.unique(b)
        self.nk[u] = self.N[u].sum(1)
        self.mx[u] = self.N[u].max(1)

    def _insert(self, P, hn):
        self.h[P] = hn
        np.add.at(self.N, (hn, self.P_c[P]), 1)
        u = np.unique(hn)
        self.nk[u] = self.N[u].sum(1)
        self.mx[u] = self.N[u].max(1)

    def _marginal_primary(self, idx, c):
        Nc = self.N[idx, c]
        if self.loss == 'hard':
            return (Nc < self.mx[idx]).astype(np.float64)
        n = self.nk[idx]
        return (
            (self.flogf[n + 1] - self.flogf[n])
            - (self.flogf[Nc + 1] - self.flogf[Nc])
        )

    def _marginal_class_cells(self, idx, c):
        return (self.N[idx, c] == 0).astype(np.float64)

    def _move(self, P, cur, vals, ntry=8):
        P = np.asarray(P, dtype=np.int64)
        old = self.h[P].copy()
        before_primary = self._primary()
        before_cells = self.class_cells()
        self._remove(P)
        base = (old - cur) % self.local_D
        pos_off = self.P_pos[P] * self.local_D
        cls = self.P_c[P][:, None]
        primary_cost = np.empty(len(vals))
        reuse_cost = np.empty(len(vals))
        chunk = max(1, 4000000 // max(1, len(P)))
        for a in range(0, len(vals), chunk):
            v = vals[a:a + chunk]
            idx = (base[:, None] + v[None, :]) % self.local_D + pos_off[:, None]
            primary_cost[a:a + chunk] = self._marginal_primary(idx, cls).sum(0)
            reuse_cost[a:a + chunk] = self._marginal_class_cells(idx, cls).sum(0)
        score = primary_cost + self.reuse_tiebreak * reuse_cost
        score = score + self.rng.random(len(vals)) * 1e-7
        for vi in np.argsort(score)[:ntry]:
            hn = (
                (base + vals[vi]) % self.local_D
                + self.P_pos[P] * self.local_D
            )
            self._insert(P, hn)
            after_primary = self._primary()
            after_cells = self.class_cells()
            accept = (after_primary < before_primary - 1e-12) or (
                abs(after_primary - before_primary) <= 1e-12 and after_cells <= before_cells
            )
            if accept:
                return int(vals[vi])
            self._remove(P)
        self._insert(P, old)
        return int(cur)

    def sweep(self, ntry=8):
        params = [('s', p) for p in self.s_params] + [('d', p) for p in self.d_params]
        self.rng.shuffle(params)
        for kind, p in params:
            if kind == 's':
                P = self.occ_s[p]
                self.s[p] = self._move(P, self.s[p], self.vals_s, ntry=ntry)
            else:
                delta, tok = p
                P = self.occ_d[p]
                row = delta + self.R
                self.d[row, tok] = self._move(P, self.d[row, tok], self.vals_d, ntry=ntry)

    def table_strings(self):
        mode = self.N.argmax(1)
        return [self.cls_str[c] for c in mode]

    def resegment(self):
        T = self.table_strings()
        changed = 0
        for e in range(self.n_ex):
            a, b = self.starts[e], self.starts[e + 1]
            L = b - a
            t = self.ex_target[e]
            n = len(t)
            tab = [T[self.h[p]] for p in range(a, b)]
            cur = [self.cls_str[c] for c in self.P_c[a:b]]
            if tab == cur:
                continue
            INF = 1e9
            dp = np.full((L + 1, n + 1), INF)
            bk = [[None] * (n + 1) for _ in range(L + 1)]
            dp[0, 0] = 0.0
            for i in range(L):
                for o in range(n + 1):
                    if dp[i, o] >= INF:
                        continue
                    for o2 in range(o, min(n, o + self.max_len) + 1):
                        piece = t[o:o2]
                        if piece not in self.cls_id:
                            continue
                        c = dp[i, o] + (piece != tab[i]) + 1e-3 * (piece != cur[i])
                        if c < dp[i + 1, o2]:
                            dp[i + 1, o2] = c
                            bk[i + 1][o2] = (o, piece)
            if bk[L][n] is None:
                continue
            o, new = n, []
            for i in range(L, 0, -1):
                o, piece = bk[i][o]
                new.append(piece)
            new.reverse()
            if new != cur:
                self.P_c[a:b] = [self.cls_id[piece] for piece in new]
                changed += 1
        self._rebuild()
        return changed

    def table_exact_match(self):
        T = self.table_strings()
        ok = []
        for e in range(self.n_ex):
            a, b = self.starts[e], self.starts[e + 1]
            ok.append(''.join(T[self.h[p]] for p in range(a, b)) == self.ex_target[e])
        return float(np.mean(ok))

    def feature_reuse(self):
        return self.NP - int(np.count_nonzero(self.N))

    def fit(self, rounds=10, sweeps=10, ntry=8, verbose=True):
        t0 = time.time()
        best = (self._primary(), self.class_cells(), self.d.copy(), self.s.copy(), self.P_c.copy())
        if verbose:
            print(f"init: primary={self._primary():.4f} ce={self.ce():.2f} class-cells={self.class_cells()}")
        for r in range(rounds):
            stalled = 0
            prev = (self._primary(), self.class_cells())
            for sw in range(sweeps):
                self.sweep(ntry=ntry)
                now = (self._primary(), self.class_cells())
                if verbose:
                    print(f"  round {r} sweep {sw}: primary={now[0]:.4f} ce={self.ce():.2f} class-cells={now[1]}")
                if now < best[:2]:
                    best = (now[0], now[1], self.d.copy(), self.s.copy(), self.P_c.copy())
                if now == prev:
                    stalled += 1
                    if stalled >= 2:
                        break
                else:
                    stalled = 0
                prev = now
            if self.do_reseg:
                ch = self.resegment()
                if verbose and ch:
                    print(f"  round {r}: resegmented {ch} examples; primary={self._primary():.4f}")
            if self.loss == 'hard' and self.conflicts() == 0:
                break
        self.d, self.s, self.P_c = best[2], best[3], best[4]
        self._rebuild()
        if verbose:
            print(f"done: primary={self._primary():.4f} conflicts={self.conflicts()} table-train-exact={self.table_exact_match():.4f} reuse={self.feature_reuse()} [{time.time() - t0:.1f}s]")
        return self

    def compile(self, verbose=True):
        br = self.brain
        D = self.D
        BIG = 10 ** 9
        br.hash_directions = self.d.copy()
        br.hash_start_positions = self.s.copy()
        T = self.table_strings()
        used = self.nk > 0
        free = ~used
        def nearest_free(x):
            for r in range(1, D):
                for y in (x - r, x + r):
                    if 0 <= y < D and free[y]:
                        return y
            raise RuntimeError("no free deflector left; increase local_D or max_input_len")
        sink = int(np.argmax(free))
        free[sink] = False
        dl = [[k, -BIG] for k in range(D)]
        need = 1
        chain_nodes = 0
        max_hop = 0
        for k in np.flatnonzero(used):
            S = T[k]
            if not S:
                continue
            nodes = [int(k)]
            for _ in S[1:]:
                u = nearest_free(nodes[-1])
                free[u] = False
                nodes.append(u)
                chain_nodes += 1
            for node, ch, nxt in zip(nodes, S, nodes[1:] + [sink]):
                dl[node] = [int(nxt), ch]
            cumulative_cost = 0
            for a, b in zip(nodes, nodes[1:]):
                cumulative_cost += abs(b - a) - 1
                max_hop = max(max_hop, abs(b - a))
            need = max(need, cumulative_cost + 1)
        br.deflectors_list = dl
        br.ball_energy = max(br.ball_energy, need)
        if verbose:
            print(f"compiled: start-buckets={int(used.sum())} chain-nodes={chain_nodes} longest-emission={max(len(s) for s in T)} longest-hop={max_hop} ball-energy={br.ball_energy}")
        return br

def train_with_restarts(cases):
      """
      Multi-start discrete optimization. Each restart gets a fresh random brain and trainer;
      the best primary loss, then lowest class-cell count, is kept.
      """
      best = None

      for r in range(CFG['restarts']):
          seed = CFG['seed'] + r
          random.seed(seed)
          np.random.seed(seed)

          brain = BallTrapBrain(
              alphabet,
              output,
              local_D=CFG['local_D'],
              max_input_len=max_input_len,
              hash_radius=CFG['hash_radius'],
              bounce=CFG['bounce'],
              cap=CFG['cap'],
              direction_range=CFG['direction_range'],
          )
          trainer = HashTrainer(
              brain,
              cases,
              loss=CFG['loss'],
              resegment=True,
              seed=seed,
              reuse_tiebreak=CFG['reuse_tiebreak'],
          )
          trainer.fit(
              rounds=CFG['rounds'],
              sweeps=CFG['sweeps'],
              ntry=CFG['ntry'],
              verbose=False,
          )

          score = (trainer._primary(), trainer.class_cells())
          print(
              f"restart {r}: primary={score[0]:.4f}  "
              f"conflicts={trainer.conflicts()}  "
              f"train-table-exact={trainer.table_exact_match():.4f}  "
              f"reuse={trainer.feature_reuse()}"
          )

          if best is None or score < best[0]:
              best = (score, trainer)

      print(
          f"selected: primary={best[0][0]:.4f}  "
          f"conflicts={best[1].conflicts()}  "
          f"train-table-exact={best[1].table_exact_match():.4f}"
      )
      return best[1]

## Train

The only architectural change is the hash: **still one lookup per ball, no layering**.


restart 0: primary=3993.2207  conflicts=2493  train-table-exact=0.0100  reuse=1028

In [ ]:
trainer = train_with_restarts(tiny_cases)
brain = trainer.compile()
print(len(brain.deflectors_list))

restart 0: primary=0.0000  conflicts=0  train-table-exact=1.0000  reuse=993
selected: primary=0.0000  conflicts=0  train-table-exact=1.0000


RuntimeError: no free deflector left; increase local_D or max_input_len

## Evaluate on the real discrete simulator

In [ ]:
tr_exact, tr_ratio = evaluate(brain, tiny_cases)
# te_exact, te_ratio = evaluate(brain, test_cases)
print(f"train: exact-match={tr_exact:.4f}  similarity={tr_ratio:.4f}")
#print(f"test : exact-match={te_exact:.4f}  similarity={te_ratio:.4f}")

print("\nTraining examples:")
for inp, ans in sorted(tiny_cases)[:50]:
    pred = brain.compute_input(inp)
    print(f"{inp!r:12} -> {pred!r:6}  target={ans!r}")

# print("\nHeld-out examples:")
# for inp, ans in sorted(test_cases)[:12]:
#     pred = brain.compute_input(inp)
#     print(f"{inp!r:12} -> {pred!r:6}  target={ans!r}")


In [ ]:

for _ in range(10):
  sentence = input("Input: ")
  sentence = brain.compute_input(sentence)
  print(sentence)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Extract deflectors details: [destination, value]
deflectors = brain.deflectors_list

# Separate into values: strings (emitted symbols) vs integers (bounce energy)
target_values = []
for dest, val in deflectors:
    if isinstance(val, int):
        if val < -1e6:
            target_values.append("Unused (Trap / -1B)")
        else:
            target_values.append(f"Int Bounce ({val})")
    else:
        target_values.append(f"Token ('{val}')")

df_targets = pd.DataFrame({'Target Category': target_values})
counts = df_targets['Target Category'].value_value_counts() if hasattr(df_targets['Target Category'], 'value_value_counts') else df_targets['Target Category'].value_counts()

# Plot distribution
plt.figure(figsize=(10, 5))
sns.barplot(x=counts.values, y=counts.index, palette='viridis')
plt.title('Distribution of Deflector Targets in the Deflector Table')
plt.xlabel('Count')
plt.ylabel('Target Category')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# We will simulate a walk for each possible starting node (0 to D-1) and record the step count.
def get_bounce_counts(brain):
    step_counts = []
    dl = brain.deflectors_list
    cap = brain.cap

    for start_node in range(brain.D):
        be = brain.ball_energy
        node = start_node
        steps = 0

        while be > 0 and steps < cap:
            dest, val = dl[node]
            dist = abs(dest - node)
            if isinstance(val, int):
                be += val - dist
            else:
                be += 1 - dist
            node = dest
            steps += 1

        step_counts.append(steps)
    return step_counts

bounce_counts = get_bounce_counts(brain)

# Plot the results
plt.figure(figsize=(12, 5))

# Left subplot: Histogram of step distributions
plt.subplot(1, 2, 1)
plt.hist(bounce_counts, bins=np.arange(0, brain.cap + 2) - 0.5, rwidth=0.8, color='skyblue', edgecolor='black')
plt.title('Distribution of Bounce Counts per Ball')
plt.xlabel('Number of Bounces (Steps)')
plt.ylabel('Number of Balls (Nodes)')
plt.xticks(range(0, brain.cap + 1))
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Right subplot: Sequence plot across all D nodes
plt.subplot(1, 2, 2)
plt.plot(bounce_counts, color='teal', alpha=0.6, linewidth=0.5)
plt.title('Bounce Counts by Node Index')
plt.xlabel('Node Index')
plt.ylabel('Bounces')
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Notes and limits

* **What changed.** The original global hash was an associative-memory address. The new
  address is a reusable local feature hash: the same `(relative position, token)` feature
  is shared across all examples, and each ball position has its own bucket region.
* **Why this can generalize.** A test context does not need to reproduce an entire training
  context. It only needs the relevant local features to land in buckets whose table entries
  were learned from other examples.
* **What did not change.** There is still exactly one lookup per ball. The walk never reads
  another input token after hashing, so this is not the proposed layered architecture.
* **Training objective.** Class purity / profiled CE remains primary. Reuse is only a
  tie-breaker, which prevents the optimizer from sacrificing fit merely to get more sharing.
* **Capacity knob.** `local_D` controls buckets per slot. Smaller values put more pressure
  on feature reuse; larger values make the search closer to memorization.
* **Locality knob.** `hash_radius` controls how much context a ball can use. Increasing it
  raises expressive power but weakens the inductive bias toward reuse.
